In [1]:
import sys
!{sys.executable} -m pip install groq rich python-dotenv pandas mcp google-adk --quiet

In [2]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


# R&D Experiment Prioritization Agent
## Kaggle Capstone — Agents for Business Track

A multi-agent system that recommends next R&D experiments using:
- **4 chained agents** (Intake → Context → Prioritization → Report)
- **MCP Server** exposing 8 tools
- **Safety guardrails** on every LLM call

In [6]:
#Setup:
# Install dependencies (uncomment if running fresh on Kaggle)
#!pip install groq rich python-dotenv pandas mcp google-adk

import os
os.environ["LLM_PROVIDER"] = "groq"
os.environ["GROQ_API_KEY"] = "your_key_here"   # paste key for submission
os.environ["GROQ_MODEL"] = "llama-3.3-70b-versatile"

# MCP tool verification:

In [2]:
from src.tools.experiment_lookup import get_all_experiments, get_top_alignment_experiments
from src.tools.constraints_tool import get_decision_rules

exps = get_all_experiments()
print(f"Total experiments: {len(exps)}")
print(f"Top alignment experiment: {get_top_alignment_experiments(1)[0]['experiment_id']}")
print(f"Decision rules: {get_decision_rules()}")

Total experiments: 12
Top alignment experiment: EXP011
Decision rules: {'minimum_stability_score': 0.5, 'target_alignment_score': 0.7, 'penalize_failed_conditions': True, 'favor_novel_but_feasible_experiments': True, 'prefer_experiments_similar_to_successful_runs': True}


In [4]:
import os
key = os.environ.get("GROQ_API_KEY", "NOT SET")
print(f"Key starts with: {key[:8]}...")
print(f"Key length: {len(key)}")

Key starts with: your_key...
Key length: 13


In [ ]:
gsk_...s9cD

# Agent 1:

In [7]:
from src.agents.intake_agent import run_intake_agent

goal = "what should we run next to improve alignment"
intake = run_intake_agent(goal)
print(f"Intent: {intake['intent']}")
print(f"Rephrased: {intake['goal_rephrased']}")

Intent: prioritize_experiments
Rephrased: What experiment should we run next to improve alignment?


# Agent 2:

In [8]:
from src.agents.context_agent import run_context_agent

context = run_context_agent(intake["goal_rephrased"])
print(context["llm_analysis"])

**Analysis of the Project Context**

The goal of the project is to improve alignment, with a target alignment score of 0.7. The experiment history summary shows that out of 12 total runs, 5 were successful, 5 were partial, and 2 failed. The best alignment was achieved in EXP011 with a score of 0.73 using a pulsed field control, but this came at the cost of stability. The best stability was achieved in EXP006 with a score of 0.69 using a low field control. The best overall trade-off was achieved in EXP012 with an alignment score of 0.67, stability score of 0.64, and a cost of 220 EUR.

**Key Constraints and Challenges**

1. **Budget constraint**: The maximum budget per run is 200 EUR, which is a significant constraint considering that the best overall trade-off experiment (EXP012) exceeded this budget.
2. **Time constraint**: The maximum time per run is 8 hours, which is another constraint that needs to be considered.
3. **Stability requirement**: The minimum stability required is 0.5, 

# Agent 3:

In [9]:
from src.agents.prioritization_agent import run_prioritization_agent

prioritization = run_prioritization_agent(context)
print(prioritization["ranking_explanation"])

**1. Why these experiments are recommended:**

These experiments are recommended because they represent the top-ranked candidates based on their alignment scores, stability, and cost. EXP012, despite being expensive, achieved the best overall trade-off. EXP010 and EXP007, both using gradient field conditions, showed promising results with a good balance between alignment and stability. They are recommended as they align with the project's goal of improving alignment while considering the constraints of budget, time, and stability.

**2. What the main trade-off for each one is:**

- **EXP012 (adaptive_field):** The main trade-off is between achieving a high alignment score (0.67) and stability (0.64) at the cost of exceeding the budget constraint (220 EUR).
- **EXP010 (gradient_field):** The main trade-off is a slightly lower stability score (0.55) compared to EXP012, but with a lower cost (175 EUR) and a higher alignment score (0.69).
- **EXP007 (gradient_field):** The trade-off is a m

# Agent 4:

In [10]:
from src.agents.report_agent import run_report_agent

memo = run_report_agent(context, prioritization)
print(memo)

**Decision Memo: Next Experiment Recommendations**

The project is currently focused on improving particle alignment without compromising stability. After evaluating various experiment candidates, we recommend the top 3 experiments: EXP012, EXP010, and EXP007. These experiments balance alignment, stability, cost, and runtime effectively.

The key trade-offs are:
- EXP012 offers high alignment and stability but at a higher cost (220 EUR).
- EXP010 provides a high alignment score but slightly lower stability at a moderate cost (175 EUR).
- EXP007 presents a balanced trade-off between alignment, stability, and cost (170 EUR).

Based on these evaluations, I suggest initiating **EXP007 (gradient_field)** first. It offers a promising alignment score (0.66) and stability (0.57) within the budget constraint, posing less financial risk while providing valuable insights. This experiment allows for the assessment of gradient field conditions at a moderate cost and time, making it an ideal startin

---
## Results Summary

The 4-agent pipeline ran successfully end-to-end:

| Step | Agent | Output |
|---|---|---|
| 1 | Intake | Validated goal, extracted intent |
| 2 | Context | Analyzed 12 experiments + constraints |
| 3 | Prioritization | Ranked top 3 candidates with trade-off reasoning |
| 4 | Report | Generated decision memo saved to `output/` |

**Key recommendation:** EXP007 (gradient_field) — best balance of alignment, 
stability, and cost within constraints.

---
*Built for the Kaggle Agents Capstone — Agents for Business Track*